# **Environment check & imports**

In [1]:
import os, shutil, random, json
import numpy as np
import cv2
from PIL import Image
import albumentations as A
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models


# **Attach datasets & prepare data**

In [2]:
WORK = "currency_data"
SOURCE_MAP = {
    "dataset": f"{WORK}/data/data"
}

RAW_MERGED = f"{WORK}/data/raw_real"
os.makedirs(RAW_MERGED, exist_ok=True)

def collect_images(src_root, dest, prefix, skip_folders=("Background",)):
    count = 0
    for dirpath, dirnames, filenames in os.walk(src_root):
        dirnames[:] = [d for d in dirnames if d not in skip_folders]
        if any(skip in dirpath for skip in skip_folders):
            continue
        for fn in filenames:
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                shutil.copy(os.path.join(dirpath, fn), os.path.join(dest, f"{prefix}_{count}_{fn}"))
                count += 1
    return count

for name, path in SOURCE_MAP.items():
    collect_images(path, RAW_MERGED, name)

FAKE_CLASSES = ["fake_print_defect", "fake_color_shift", "fake_missing_thread", "fake_missing_microprint"]

DATA_ROOT = f"{WORK}/data/processed_all"
for cls in ["real"] + FAKE_CLASSES:
    os.makedirs(f"{DATA_ROOT}/{cls}", exist_ok=True)

def make_print_defect(img):
    t = A.Compose([
        A.GaussianBlur(blur_limit=(5, 9), p=1.0),
        A.Downscale(scale_range=(0.4, 0.6), p=1.0),
        A.ImageCompression(quality_range=(25, 45), p=1.0),
    ])
    return t(image=img)["image"]

def make_color_shift(img):
    t = A.Compose([
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=15, p=1.0),
        A.RGBShift(r_shift_limit=25, g_shift_limit=25, b_shift_limit=25, p=1.0),
    ])
    return t(image=img)["image"]

def make_missing_thread(img):
    h, w = img.shape[:2]
    x1, x2 = int(w * 0.35), int(w * 0.42)
    mask = np.zeros((h, w), dtype=np.uint8)
    mask[:, x1:x2] = 255
    return cv2.inpaint(img, mask, 7, cv2.INPAINT_TELEA)

def make_missing_microprint(img):
    h, w = img.shape[:2]
    x1, y1, x2, y2 = int(w*0.55), int(h*0.6), int(w*0.85), int(h*0.75)
    region = img[y1:y2, x1:x2]
    img[y1:y2, x1:x2] = cv2.GaussianBlur(region, (15, 15), 0)
    return img

GENERATORS = {
    "fake_print_defect": make_print_defect,
    "fake_color_shift": make_color_shift,
    "fake_missing_thread": make_missing_thread,
    "fake_missing_microprint": make_missing_microprint,
}

real_files = os.listdir(RAW_MERGED)
random.shuffle(real_files)

for fn in real_files:
    shutil.copy(os.path.join(RAW_MERGED, fn), f"{DATA_ROOT}/real/{fn}")

n_per_class = max(1, len(real_files) // len(FAKE_CLASSES))
idx = 0
for cls, gen_fn in GENERATORS.items():
    for fn in real_files[idx: idx + n_per_class]:
        img_path = os.path.join(RAW_MERGED, fn)
        img = cv2.imread(img_path)
        if img is None or img.size == 0:
            continue
        try:
            fake_img = gen_fn(img.copy())
            cv2.imwrite(f"{DATA_ROOT}/{cls}/{fn}", fake_img)
        except Exception:
            pass
    idx += n_per_class


# **Train/val split**

In [3]:
SPLIT_ROOT = f"{WORK}/data/split"
splits = ["train", "val", "test"]
classes = ["real"] + FAKE_CLASSES

for s in splits:
    for c in classes:
        os.makedirs(f"{SPLIT_ROOT}/{s}/{c}", exist_ok=True)

for c in classes:
    files = os.listdir(f"{DATA_ROOT}/{c}")
    random.shuffle(files)
    n = len(files)
    n_train, n_val = max(1, int(n * 0.7)), max(1, int(n * 0.15))
    split_map = {
        "train": files[:n_train],
        "val": files[n_train:n_train + n_val],
        "test": files[n_train + n_val:],
    }
    for s, flist in split_map.items():
        for fn in flist:
            shutil.copy(f"{DATA_ROOT}/{c}/{fn}", f"{SPLIT_ROOT}/{s}/{c}/{fn}")


# **Build tf.data datasets**

In [4]:
IMG_SIZE = (224, 224)
BATCH = 8

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_ROOT}/train", image_size=IMG_SIZE, batch_size=BATCH, label_mode="int")
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_ROOT}/val", image_size=IMG_SIZE, batch_size=BATCH, label_mode="int")

CLASS_NAMES = list(train_ds.class_names)
with open(f"{WORK}/class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)

augment = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
])

train_ds = train_ds.map(lambda x, y: (augment(x, training=True), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)


# **Build, train & export model**

In [5]:
base = EfficientNetB0(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(len(CLASS_NAMES), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy"
)

model.fit(train_ds, validation_data=val_ds, epochs=1)

os.makedirs("models", exist_ok=True)
model.save_weights("models/currency_model.h5")
model.save_weights("currency_model.h5")
model.save_weights("best_model_stage1.h5")
model.save_weights("best_model_finetuned.h5")
